In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("flight_data_2024.csv")
df.head()

/tmp/ipykernel_43981/3464087177.py:1: DtypeWarning: Columns (0: cancellation_code) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("flight_data_2024.csv")


,year,month,day_of_month,day_of_week,fl_date,op_unique_carrier,op_carrier_fl_num,origin,origin_city_name,origin_state_nm,...,diverted,crs_elapsed_time,actual_elapsed_time,air_time,distance,carrier_delay,weather_delay,nas_delay,security_delay,late_aircraft_delay
0,2024,1,1,1,2024-01-01,9E,4814.0,JFK,"New York, NY",New York,...,0,136.0,122.0,84.0,509.0,0,0,0,0,0
1,2024,1,1,1,2024-01-01,9E,4815.0,MSP,"Minneapolis, MN",Minnesota,...,0,130.0,114.0,88.0,622.0,0,0,0,0,0
2,2024,1,1,1,2024-01-01,9E,4817.0,JFK,"New York, NY",New York,...,0,106.0,90.0,61.0,288.0,0,0,0,0,0
3,2024,1,1,1,2024-01-01,9E,4817.0,RIC,"Richmond, VA",Virginia,...,0,111.0,76.0,51.0,288.0,0,0,0,0,0
4,2024,1,1,1,2024-01-01,9E,4818.0,DTW,"Detroit, MI",Michigan,...,0,79.0,70.0,45.0,237.0,0,0,0,0,0


In [7]:
airports= df['origin'].unique().tolist()
print(airports)

['JFK', 'MSP', 'RIC', 'DTW', 'JAX', 'LGA', 'CHS', 'ITH', 'CLE', 'GRR', 'CMH', 'ATL', 'TLH', 'CLT', 'PVD', 'MIA', 'RDU', 'BTV', 'STL', 'SAV', 'BDL', 'IND', 'IAD', 'PHL', 'DHN', 'BGR', 'EYW', 'TYS', 'GTR', 'PNS', 'ILM', 'MCI', 'FAY', 'SYR', 'TUL', 'MLU', 'RST', 'MLI', 'DAY', 'ROC', 'LEX', 'AEX', 'BNA', 'MDT', 'MYR', 'MQT', 'DSM', 'OAJ', 'CHO', 'BHM', 'ICT', 'SDF', 'ATW', 'PWM', 'JAN', 'CHA', 'CWA', 'OMA', 'MSN', 'CRW', 'TRI', 'HPN', 'LFT', 'CID', 'LIT', 'CVG', 'MEM', 'ROA', 'GRB', 'GSP', 'MOB', 'SGF', 'BTR', 'ALB', 'XNA', 'GFK', 'AVL', 'HSV', 'FLL', 'GSO', 'ABE', 'ORF', 'VLD', 'BUF', 'LAN', 'AZO', 'AGS', 'BOS', 'MKE', 'BWI', 'BMI', 'BGM', 'CAE', 'RSW', 'MGM', 'DFW', 'LAS', 'AUS', 'PHX', 'SJC', 'HNL', 'TUS', 'AVP', 'LAX', 'DCA', 'ORD', 'MFE', 'BFL', 'SLC', 'ELP', 'BOI', 'MCO', 'AMA', 'OGG', 'SJU', 'MSY', 'TPA', 'STT', 'EWR', 'SMF', 'SFO', 'BUR', 'ABQ', 'PBI', 'PDX', 'ANC', 'DEN', 'PIT', 'SAT', 'STX', 'FSD', 'IAH', 'RNO', 'SNA', 'ONT', 'COS', 'SRQ', 'ECP', 'SEA', 'BZN', 'PSP', 'FAT', 'EGE'

In [7]:
%pip install airportsdata openmeteo-requests requests-cache retry-requests pandas

Note: you may need to restart the kernel to use updated packages.


In [8]:
%pip install requests_cache

Note: you may need to restart the kernel to use updated packages.


In [12]:
import airportsdata
import openmeteo_requests
import requests_cache
import pandas as pd
import time
import os
from retry_requests import retry

# =====================================================================
# STEP 1: Setup caching and retry client for Open-Meteo
# =====================================================================
cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

# =====================================================================
# STEP 2: Load airport database and match coordinates
# =====================================================================
airports_db = airportsdata.load('IATA')

airport_codes = ['JFK', 'MSP', 'RIC', 'DTW', 'JAX', 'LGA', 'CHS', 'ITH', 'CLE', 'GRR', 'CMH', 'ATL', 'TLH', 'CLT', 'PVD', 'MIA', 'RDU', 'BTV', 'STL', 'SAV', 'BDL', 'IND', 'IAD', 'PHL', 'DHN', 'BGR', 'EYW', 'TYS', 'GTR', 'PNS', 'ILM', 'MCI', 'FAY', 'SYR', 'TUL', 'MLU', 'RST', 'MLI', 'DAY', 'ROC', 'LEX', 'AEX', 'BNA', 'MDT', 'MYR', 'MQT', 'DSM', 'OAJ', 'CHO', 'BHM', 'ICT', 'SDF', 'ATW', 'PWM', 'JAN', 'CHA', 'CWA', 'OMA', 'MSN', 'CRW', 'TRI', 'HPN', 'LFT', 'CID', 'LIT', 'CVG', 'MEM', 'ROA', 'GRB', 'GSP', 'MOB', 'SGF', 'BTR', 'ALB', 'XNA', 'GFK', 'AVL', 'HSV', 'FLL', 'GSO', 'ABE', 'ORF', 'VLD', 'BUF', 'LAN', 'AZO', 'AGS', 'BOS', 'MKE', 'BWI', 'BMI', 'BGM', 'CAE', 'RSW', 'MGM', 'DFW', 'LAS', 'AUS', 'PHX', 'SJC', 'HNL', 'TUS', 'AVP', 'LAX', 'DCA', 'ORD', 'MFE', 'BFL', 'SLC', 'ELP', 'BOI', 'MCO', 'AMA', 'OGG', 'SJU', 'MSY', 'TPA', 'STT', 'EWR', 'SMF', 'SFO', 'BUR', 'ABQ', 'PBI', 'PDX', 'ANC', 'DEN', 'PIT', 'SAT', 'STX', 'FSD', 'IAH', 'RNO', 'SNA', 'ONT', 'COS', 'SRQ', 'ECP', 'SEA', 'BZN', 'PSP', 'FAT', 'EGE', 'EUG', 'OKC', 'SAN', 'GUC', 'HDN', 'VPS', 'MRY', 'GEG', 'MSO', 'SBP', 'MTJ', 'LIH', 'SBA', 'DAB', 'JAC', 'KOA', 'PSC', 'FAI', 'STS', 'OAK', 'JNU', 'PAE', 'DAL', 'KTN', 'SIT', 'RDM', 'BQN', 'ORH', 'PSE', 'MDW', 'TVC', 'HOU', 'HRL', 'LGB', 'GPT', 'FAR', 'GNV', 'FCA', 'MLB', 'BIS', 'BIL', 'ISP', 'TTN', 'PIE', 'IAG', 'USA', 'SFB', 'BLV', 'RFD', 'FWA', 'LCK', 'PGD', 'FNT', 'PVU', 'MOT', 'PBG', 'AZA', 'TOL', 'PIA', 'SBN', 'BLI', 'PSM', 'CKB', 'LRD', 'RAP', 'SMX', 'IDA', 'GRI', 'SCK', 'SPI', 'ITO', 'PPG', 'COU', 'LBB', 'ACT', 'SHV', 'MAF', 'GRK', 'LAW', 'MHK', 'ABI', 'GJT', 'CRP', 'HHH', 'SJT', 'EVV', 'BRO', 'CMI', 'CLL', 'SAF', 'FSM', 'BPT', 'ACY', 'LBE', 'CAK', 'MHT', 'ASE', 'SGU', 'GCK', 'LCH', 'SWO', 'MFR', 'ROW', 'TYR', 'TXK', 'GGG', 'FLG', 'DRO', 'YUM', 'HLN', 'ALW', 'SUN', 'DLH', 'MBS', 'PIH', 'ABY', 'BQK', 'LWS', 'TWF', 'GTF', 'ELM', 'CSG', 'CIU', 'HIB', 'ABR', 'ESC', 'INL', 'CPR', 'BRD', 'APN', 'PLN', 'RHI', 'IMT', 'XWA', 'CDC', 'BJI', 'EKO', 'BTM', 'ACV', 'DIK', 'LNK', 'CYS', 'CMX', 'PIB', 'JST', 'CNY', 'HYS', 'SLN', 'VCT', 'LAR', 'DEC', 'DVL', 'JMS', 'JLN', 'LBF', 'FOD', 'MCW', 'MEI', 'PRC', 'VEL', 'GCC', 'SUX', 'DDC', 'LBL', 'BFF', 'RIW', 'RKS', 'SHR', 'SCE', 'BIH', 'RDD', 'OTH', 'SPS', 'GUM', 'SPN', 'OME', 'OTZ', 'BET', 'SCC', 'CDV', 'YAK', 'PSG', 'WRG', 'ADQ', 'SWF', 'HTS', 'HGR', 'STC', 'BRW', 'ADK', 'LSE', 'AKN', 'WYS', 'ACK', 'HYA', 'COD', 'MVY', 'HOB', 'GST', 'DLG', 'EWN', 'PQI', 'EAR', 'MGW', 'EAU']

valid_airports, lats, lons = [], [], []

for code in airport_codes:
    if code in airports_db:
        valid_airports.append(code)
        lats.append(airports_db[code]['lat'])
        lons.append(airports_db[code]['lon'])
    else:
        print(f"Warning: Could not find coordinates for: {code}")

# =====================================================================
# STEP 3 & 4: Fetch Data, Handle Limits, and Auto-Save to CSV
# =====================================================================
url = "https://archive-api.open-meteo.com/v1/archive"
batch_size = 5 
csv_filename = 'airports_weather_2024.csv'

# Delete old file if restarting from scratch to prevent duplicates
if os.path.exists(csv_filename):
    os.remove(csv_filename)

print(f"\nFetching weather for {len(valid_airports)} airports through 2024...")
print(f"Data will automatically save to {csv_filename} after every batch!\n")

write_header = True # Writes columns on the first pass only

for i in range(0, len(valid_airports), batch_size):
    batch_airports = valid_airports[i:i+batch_size]
    
    params = {
        "latitude": lats[i:i+batch_size],
        "longitude": lons[i:i+batch_size],
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "hourly": ["temperature_2m", "precipitation", "snowfall", "weather_code", "wind_speed_10m", "wind_gusts_10m"],
        "timezone": "auto" 
    }
    
    # --- Self-Healing Retry Loop ---
    success = False
    while not success:
        try:
            responses = openmeteo.weather_api(url, params=params)
            success = True 
            
        except Exception as e:
            error_msg = str(e)
            if "Minutely" in error_msg:
                print(f"\n⏳ Minutely limit hit at batch {i}. Pausing for 60 seconds...")
                time.sleep(60)
            elif "Hourly" in error_msg:
                current_time = pd.Timestamp.now().strftime('%H:%M:%S')
                print(f"\n🛑 Hourly limit hit! Time is {current_time}. Sleeping for 61 mins. Do not close notebook...")
                time.sleep(3660) 
            else:
                raise e 
    
    # --- Data Processing ---
    batch_weather_data = [] 
    
    for idx, response in enumerate(responses):
        airport = batch_airports[idx]
        hourly = response.Hourly()
        
        utc_time = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )
        local_time = (utc_time + pd.Timedelta(seconds=response.UtcOffsetSeconds())).tz_localize(None)
        
        df_airport = pd.DataFrame({
            "airport": airport,
            "datetime": local_time,
            "temperature_2m": hourly.Variables(0).ValuesAsNumpy(),
            "precipitation": hourly.Variables(1).ValuesAsNumpy(),
            "snowfall": hourly.Variables(2).ValuesAsNumpy(),
            "weather_code": hourly.Variables(3).ValuesAsNumpy(),
            "wind_speed_10m": hourly.Variables(4).ValuesAsNumpy(),
            "wind_gusts_10m": hourly.Variables(5).ValuesAsNumpy()
        })
        
        df_airport['fl_date'] = df_airport['datetime'].dt.strftime('%Y-%m-%d')
        df_airport['time_hhmm'] = df_airport['datetime'].dt.strftime('%H%M')
        
        batch_weather_data.append(df_airport)

    # --- Append to CSV ---
    batch_df = pd.concat(batch_weather_data, ignore_index=True)
    batch_df = batch_df[['airport', 'fl_date', 'time_hhmm', 'temperature_2m', 'precipitation', 'snowfall', 'weather_code', 'wind_speed_10m', 'wind_gusts_10m']]
    
    batch_df.to_csv(csv_filename, mode='a', index=False, header=write_header)
    write_header = False 
    
    # --- Rest for limits ---
    processed_count = min(i + batch_size, len(valid_airports))
    print(f"Processed & Saved {processed_count} / {len(valid_airports)} airports. Resting 15s...")
    time.sleep(15)

print(f"\nAll airports processed safely! You can load the data anytime using pd.read_csv('{csv_filename}')")


Fetching weather for 348 airports through 2024...
Data will automatically save to airports_weather_2024.csv after every batch!

Processed & Saved 5 / 348 airports. Resting 15s...
Processed & Saved 10 / 348 airports. Resting 15s...
Processed & Saved 15 / 348 airports. Resting 15s...
Processed & Saved 20 / 348 airports. Resting 15s...
Processed & Saved 25 / 348 airports. Resting 15s...
Processed & Saved 30 / 348 airports. Resting 15s...
Processed & Saved 35 / 348 airports. Resting 15s...
Processed & Saved 40 / 348 airports. Resting 15s...
Processed & Saved 45 / 348 airports. Resting 15s...
Processed & Saved 50 / 348 airports. Resting 15s...
Processed & Saved 55 / 348 airports. Resting 15s...
Processed & Saved 60 / 348 airports. Resting 15s...
Processed & Saved 65 / 348 airports. Resting 15s...
Processed & Saved 70 / 348 airports. Resting 15s...
Processed & Saved 75 / 348 airports. Resting 15s...
Processed & Saved 80 / 348 airports. Resting 15s...
Processed & Saved 85 / 348 airports. Res

In [15]:
import pandas as pd
import time
import os

url = "https://archive-api.open-meteo.com/v1/archive"
batch_size = 5 
csv_filename = 'airports_pressure_2024.csv' # Saving to a NEW file

if os.path.exists(csv_filename):
    os.remove(csv_filename)

print("Fetching pressure data...")
write_header = True 

for i in range(0, len(valid_airports), batch_size):
    batch_airports = valid_airports[i:i+batch_size]
    
    params = {
        "latitude": lats[i:i+batch_size],
        "longitude": lons[i:i+batch_size],
        "start_date": "2024-01-01",
        "end_date": "2024-12-31",
        "hourly": ["surface_pressure", "pressure_msl"], # Fetching ONLY pressure
        "timezone": "auto" 
    }
    
    success = False
    while not success:
        try:
            responses = openmeteo.weather_api(url, params=params)
            success = True 
        except Exception as e:
            error_msg = str(e)
            if "Minutely" in error_msg:
                time.sleep(60)
            elif "Hourly" in error_msg:
                time.sleep(3660) 
            elif "Daily" in error_msg:
                print("🛑 DAILY LIMIT HIT! You must wait 24 hours to run this again.")
                raise e
            else:
                raise e 
    
    batch_pressure_data = [] 
    
    for idx, response in enumerate(responses):
        airport = batch_airports[idx]
        hourly = response.Hourly()
        
        utc_time = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left"
        )
        local_time = (utc_time + pd.Timedelta(seconds=response.UtcOffsetSeconds())).tz_localize(None)
        
        df_airport = pd.DataFrame({
            "airport": airport,
            "datetime": local_time,
            "surface_pressure": hourly.Variables(0).ValuesAsNumpy(),
            "pressure_msl": hourly.Variables(1).ValuesAsNumpy()
        })
        
        df_airport['fl_date'] = df_airport['datetime'].dt.strftime('%Y-%m-%d')
        df_airport['time_hhmm'] = df_airport['datetime'].dt.strftime('%H%M')
        
        batch_pressure_data.append(df_airport)

    batch_df = pd.concat(batch_pressure_data, ignore_index=True)
    batch_df = batch_df[['airport', 'fl_date', 'time_hhmm', 'surface_pressure', 'pressure_msl']]
    
    batch_df.to_csv(csv_filename, mode='a', index=False, header=write_header)
    write_header = False 
    time.sleep(15)

print(f"Success! Data saved to {csv_filename}")

Fetching pressure data...
Success! Data saved to airports_pressure_2024.csv
